<a href="https://colab.research.google.com/github/jgromero/drl-csic/blob/es/code/from-scratch/cartpole-reinforce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# REINFORCE para _cartpole_

[**Juan Gómez Romero**](https://ccia.ugr.es/~jgomez)  
Departamento de Ciencias de la Computación e Inteligencia Artificial  
Universidad de Granada  
Este trabajo está bajo licencia [Apache License 2.0](https://choosealicense.com/licenses/apache-2.0/).

---
Basado en:
> Udacity, Deep Reinforcement Learning Course. Disponible en [GitHub](https://github.com/udacity/deep-reinforcement-learning/tree/master/reinforce).

## Configuración

In [ ]:
%pip install "gymnasium[classic-control]>=1.1"

In [ ]:
# modo de renderizado 'rgb_array': el entorno genera imágenes que se muestran como animaciones en el cuaderno
render_mode = "rgb_array"

## Entorno

Usaremos [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/), un problema de control sencillo.

<img src="https://github.com/jgromero/drl-csic/blob/es/img/cartpole.gif?raw=true"/>


Los espacios de acciones y de observaciones se pueden mostrar como sigue.

In [ ]:
import gymnasium as gym
env = gym.make("CartPole-v1", render_mode = render_mode)
print(env.observation_space)
print(env.action_space)

Para ver cómo actúa un agente, la función `show_episode()` ejecuta un episodio completo y muestra las imágenes generadas por el entorno (`render_mode = "rgb_array"`) como una animación, tanto en Google Colab como en local. La usaremos para comparar el comportamiento del agente antes y después del entrenamiento.

In [ ]:
import io
from PIL import Image
from IPython.display import display, Image as IPImage

def show_episode(env, act, max_t=500):
    """Ejecuta un episodio y lo muestra en el cuaderno como una animación.

    Parámetros
    ==========
        env: entorno creado con render_mode="rgb_array"
        act (callable): función que devuelve la acción para un estado
        max_t (int): número máximo de pasos del episodio
    """
    state, _ = env.reset()
    frames = [Image.fromarray(env.render())]
    score = 0
    for t in range(max_t):
        state, reward, terminated, truncated, _ = env.step(act(state))
        frames.append(Image.fromarray(env.render()))
        score += reward
        if terminated or truncated:
            break

    # guardar las imágenes como un GIF animado y mostrarlo
    gif = io.BytesIO()
    frames[0].save(gif, format='gif', save_all=True, append_images=frames[1:],
                   duration=1000 // env.metadata['render_fps'], loop=0)
    display(IPImage(data=gif.getvalue(), format='gif'))
    print('Puntuación:', score)

Veamos, por ejemplo, un episodio de un agente que elige acciones al azar.

In [ ]:
show_episode(env, lambda state: env.action_space.sample())

## Algoritmo REINFORCE
A continuación proporcionamos una implementación genérica de REINFORCE. Seguimos la misma estructura que en [cartpole-dqn.ipynb](https://github.com/jgromero/drl-csic/blob/es/code/from-scratch/cartpole-dqn.ipynb) pero, como el código es mucho más corto, lo incluimos en el cuaderno en lugar de en ficheros externos:

*   `Policy`: Implementación de la red neuronal del agente (la red de política), equivalente a [model.py](https://github.com/jgromero/drl-csic/blob/es/code/from-scratch/model.py) en DQN.
*   `Agent`: Implementación de las capacidades del agente (`act`, `step`, `learn`), equivalente a [dqn_agent.py](https://github.com/jgromero/drl-csic/blob/es/code/from-scratch/dqn_agent.py) en DQN.

El entorno [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/) se considera resuelto cuando el agente obtiene una puntuación media de al menos $475$ puntos en 100 episodios consecutivos. Los episodios se truncan a los $500$ pasos, que es por tanto la puntuación máxima.

### Importaciones e inicializaciones

In [ ]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np
from collections import deque
import matplotlib.pyplot as plt
%matplotlib inline

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
def plot_scores(scores, title, solved_score=None, window=100):
    """Representa la puntuación de cada episodio de entrenamiento y la media móvil de los últimos `window`.

    Parámetros
    ==========
        scores (list): puntuación de cada episodio de entrenamiento
        title (str): título de la gráfica
        solved_score (float): puntuación media para considerar resuelto el problema (opcional)
        window (int): número de episodios de la media móvil
    """
    scores = np.asarray(scores)
    avg = [np.mean(scores[max(0, i-window+1):i+1]) for i in range(len(scores))]
    plt.plot(np.arange(len(scores)), scores, alpha=0.4, label='Puntuación')
    plt.plot(np.arange(len(scores)), avg, label='Puntuación media (últimos {})'.format(window))
    if solved_score is not None:
        plt.axhline(solved_score, color='gray', linestyle='--', label='Resuelto')
    plt.ylabel('Puntuación')
    plt.xlabel('Episodio nº')
    plt.title(title)
    plt.legend()
    plt.show()

### Red de política
A diferencia de la red Q de DQN, que estima el valor de cada acción, la red de política devuelve directamente la probabilidad de seleccionar cada acción.

In [ ]:
class Policy(nn.Module):
    """Red de política: asigna a cada estado las probabilidades de las acciones."""

    def __init__(self, state_size, action_size, seed, fc1_units=16):
        """Inicializa los parámetros y construye el modelo.
        Parámetros
        ==========
            state_size (int): dimensión de cada estado
            action_size (int): dimensión de cada acción
            seed (int): semilla aleatoria
            fc1_units (int): número de nodos de la capa oculta
        """
        super(Policy, self).__init__()
        torch.manual_seed(seed)
        self.fc1 = nn.Linear(state_size, fc1_units)
        self.fc2 = nn.Linear(fc1_units, action_size)

    def forward(self, state):
        """Construye una red que asigna estado -> probabilidades de las acciones."""
        x = F.relu(self.fc1(state))
        return F.softmax(self.fc2(x), dim=1)

### Agente
La principal diferencia con el agente DQN es _cuándo_ aprende. El agente DQN aprende en cada paso a partir de un minilote de experiencias pasadas almacenadas en la memoria de experiencias (_replay buffer_). El agente REINFORCE solo almacena el logaritmo de la probabilidad de cada acción y la recompensa obtenida (`step`), y aprende al final de cada episodio (`learn`), cuando ya puede calcularse el retorno descontado desde cada paso en adelante, $G_t$.

Para reducir la varianza de la estimación del gradiente, cada acción se pondera por el $G_t$ (normalizado) en lugar de por el retorno total del episodio.

In [ ]:
GAMMA = 0.99            # factor de descuento (gamma)
LR = 1e-2               # tasa de aprendizaje (eta)

class Agent():
    """Interactúa con el entorno y aprende de él."""

    def __init__(self, state_size, action_size, seed):
        """Inicializa un objeto Agent.

        Parámetros
        ==========
            state_size (int): dimensión de cada estado
            action_size (int): dimensión de cada acción
            seed (int): semilla aleatoria
        """
        self.state_size = state_size
        self.action_size = action_size
        random.seed(seed)

        # Red de política
        self.policy = Policy(state_size, action_size, seed).to(device)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=LR)

        # Trayectoria del episodio actual
        self.saved_log_probs = []  # logaritmos de las probabilidades de las acciones tomadas
        self.rewards = []          # recompensas obtenidas

    def act(self, state):
        """Muestrea una acción para el estado dado según la política actual.

        Parámetros
        ==========
            state (array_like): estado actual
        Devuelve
        ========
            action (int): acción muestreada
            log_prob (torch.Tensor): logaritmo de la probabilidad de la acción (necesario para aprender)
        """
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.policy(state).cpu()
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)

    def step(self, log_prob, reward):
        """Almacena el logaritmo de la probabilidad de la acción tomada (At) y la recompensa obtenida (Rt+1).

        Parámetros
        ==========
            log_prob (torch.Tensor): logaritmo de la probabilidad de la acción tomada
            reward (float): recompensa obtenida
        """
        self.saved_log_probs.append(log_prob)
        self.rewards.append(reward)

    def learn(self, gamma=GAMMA):
        """Actualiza los parámetros de la política con la trayectoria del episodio que acaba de terminar.

        Parámetros
        ==========
            gamma (float): factor de descuento
        """
        # calcular el retorno descontado desde cada paso en adelante: G_t = R_t+1 + gamma * G_t+1
        returns = deque()
        G = 0
        for reward in reversed(self.rewards):
            G = reward + gamma * G
            returns.appendleft(G)

        # normalizar los retornos (reduce la varianza de la estimación del gradiente)
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-9)

        # calcular la pérdida y actualizar
        policy_loss = []
        for log_prob, G in zip(self.saved_log_probs, returns):
            policy_loss.append(-log_prob * G)
        policy_loss = torch.cat(policy_loss).sum()

        self.optimizer.zero_grad()
        policy_loss.backward()
        self.optimizer.step()

        # empezar una nueva trayectoria
        self.saved_log_probs = []
        self.rewards = []

### Bucle de entrenamiento
El bucle de entrenamiento es casi igual que en DQN. La única diferencia es la llamada a `agent.learn()` al final de cada episodio.

In [ ]:
SOLVED_SCORE = 475.0   # puntuación media (últimos 100 episodios) para considerar resuelto CartPole-v1

def reinforce(agent, n_episodes=1000, max_t=500, print_every=100):
    """REINFORCE.

    Parámetros
    ==========
        agent (Agent): agente a entrenar
        n_episodes (int): número máximo de episodios
        max_t (int): número máximo de pasos por episodio
        print_every (int): mostrar la puntuación cada print_every episodios
    """
    scores = []                          # puntuaciones de todos los episodios
    scores_window = deque(maxlen=100)    # puntuaciones de los últimos 100 episodios

    for i_episode in range(1, n_episodes+1):
        state, _ = env.reset()
        score = 0

        for t in range(max_t):

            # elegir la acción At con la red de política
            action, log_prob = agent.act(state)

            # aplicar At y obtener Rt+1, St+1
            next_state, reward, terminated, truncated, _ = env.step(action)

            # almacenar el logaritmo de la probabilidad de At y Rt+1
            agent.step(log_prob, reward)

            # avanzar al siguiente estado
            state = next_state
            score += reward

            if terminated or truncated:
                break

        # entrenar y actualizar con la trayectoria completa
        agent.learn()

        scores_window.append(score)       # guardar la última puntuación (ventana de 100)
        scores.append(score)              # guardar la última puntuación (todos los episodios)

        # mostrar información del entrenamiento
        if i_episode % print_every == 0:
            print('Episodio {}\tPuntuación media (últimos 100): {:.2f}'.format(i_episode, np.mean(scores_window)))
        if len(scores_window) == 100 and np.mean(scores_window) >= SOLVED_SCORE:
            print('\n¡Problema resuelto en {:d} episodios!\tPuntuación media (últimos 100): {:.2f}'.format(i_episode-100, np.mean(scores_window)))
            break
    else:
        print('\nProblema no resuelto en {:d} episodios.\tPuntuación media (últimos 100): {:.2f}'.format(n_episodes, np.mean(scores_window)))

    torch.save(agent.policy.state_dict(), 'checkpoint-cartpole-reinforce.pth') # guardar los pesos del agente entrenado

    return scores

agent = Agent(state_size=4, action_size=2, seed=0)
scores = reinforce(agent, n_episodes=5000)

plot_scores(scores, 'REINFORCE', SOLVED_SCORE)

## Visualizar el agente entrenado

Cargamos los pesos en el agente y lo ejecutamos durante un episodio, muestreando las acciones de la política aprendida. Las imágenes se muestran como una animación, tanto en Google Colab como en local.

In [ ]:
# cargar los pesos desde `checkpoint-cartpole-reinforce.pth`
agent_load = Agent(state_size=4, action_size=2, seed=0)

agent_load.policy.load_state_dict(torch.load('checkpoint-cartpole-reinforce.pth', map_location=device, weights_only=True))

show_episode(env, lambda state: agent_load.act(state)[0])   # act() devuelve la acción y su log-probabilidad